In [2]:
import os
from pathlib import Path

ROOT_DIR_BRAINTREEBANK = Path(r"C:\Users\simon\PyCharmMiscProject\neuroprobe-dev\braintreebank")
os.environ["ROOT_DIR_BRAINTREEBANK"] = str(ROOT_DIR_BRAINTREEBANK)

assert ROOT_DIR_BRAINTREEBANK.exists(), ROOT_DIR_BRAINTREEBANK

In [3]:
import json
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch

from sklearn.linear_model import Ridge, LinearRegression
from sklearn.metrics import r2_score, accuracy_score, roc_auc_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

import neuroprobe
from neuroprobe import (
    BrainTreebankSubject,
    BrainTreebankSubjectTrialBenchmarkDataset,
    generate_splits_cross_session,
    generate_splits_cross_subject,
    generate_splits_within_session,
)

In [4]:
import neuroprobe.config as neuroprobe_config

print("BrainTreebank root:", ROOT_DIR_BRAINTREEBANK)
print("Neuroprobe ROOT_DIR:", neuroprobe_config.ROOT_DIR)
print("Sampling rate:", neuroprobe_config.SAMPLING_RATE, "Hz")

# Subject setup
subject_id = 1
trial_id = 1
coordinates_type = "mni"  # "mni", "mni305", "cortical", "lpi"

subject = BrainTreebankSubject(
    subject_id=subject_id,
    allow_corrupted=False,
    cache=True,
    dtype=torch.float32,
    coordinates_type=coordinates_type,
)
print("Loaded subject", subject_id)
print("First 10 electrode labels:", subject.electrode_labels[:10])
print("First 10 electrode MNI coordinates:")
print(subject.get_electrode_coordinates()[:10])

# Benchmark dataset setup
eval_name = "volume"
output_indices = False
start_neural_data_before_word_onset = 0
end_neural_data_after_word_onset = neuroprobe_config.SAMPLING_RATE * 1  # 1 second

benchmark_dataset = BrainTreebankSubjectTrialBenchmarkDataset(
    subject,
    trial_id,
    dtype=torch.float32,
    eval_name=eval_name,
    output_indices=output_indices,
    start_neural_data_before_word_onset=start_neural_data_before_word_onset,
    end_neural_data_after_word_onset=end_neural_data_after_word_onset,
    lite=True,
)

data_electrode_labels = benchmark_dataset.electrode_labels
data_electrode_coordinates = benchmark_dataset.electrode_coordinates

print("Dataset type:", type(benchmark_dataset))
print("Dataset length:", len(benchmark_dataset))

first_item = benchmark_dataset[0]
print("First item type:", type(first_item))
print("First item:", first_item)

if isinstance(first_item, dict):
    print("First item data shape:", first_item["data"].shape)
    print("First item label:", first_item["label"])
else:
    print("First item shape:", first_item[0].shape)
    print("First item label:", first_item[1])

print("Number of electrodes in dataset:", len(data_electrode_labels))

BrainTreebank root: C:\Users\simon\PyCharmMiscProject\neuroprobe-dev\braintreebank
Neuroprobe ROOT_DIR: C:\Users\simon\PyCharmMiscProject\neuroprobe-dev\braintreebank
Sampling rate: 2048 Hz
Loaded subject 1
First 10 electrode labels: ['F3aOFa2', 'F3aOFa3', 'F3aOFa4', 'F3aOFa7', 'F3aOFa8', 'F3aOFa9', 'F3aOFa10', 'F3aOFa11', 'F3aOFa12', 'F3aOFa13']
First 10 electrode MNI coordinates:
tensor([[  8.0828,  44.3820, -15.1744],
        [ 12.4152,  43.6956, -14.8598],
        [ 15.7043,  42.0086, -13.4021],
        [ 27.5911,  38.1577,  -9.0090],
        [ 30.7824,  37.5062,  -7.6428],
        [ 35.1301,  35.9065,  -6.1294],
        [ 38.4192,  34.2195,  -4.6717],
        [ 42.6692,  33.6553,  -3.2497],
        [ 45.9583,  31.9683,  -1.7920],
        [ 50.2907,  31.2818,  -1.4774]])
Dataset type: <class 'neuroprobe.datasets.BrainTreebankSubjectTrialBenchmarkDataset'>
Dataset length: 3500
First item type: <class 'dict'>
First item: {'data': tensor([[ 32.9645,  27.3818,  20.4699,  ...,   2.1267,

In [5]:
import neuroprobe.train_test_splits as neuroprobe_train_test_splits

folds = neuroprobe_train_test_splits.generate_splits_within_session(
    subject,
    trial_id,
    eval_name,
    dtype=torch.float32,
    output_indices=output_indices,
    start_neural_data_before_word_onset=start_neural_data_before_word_onset,
    end_neural_data_after_word_onset=end_neural_data_after_word_onset,
    lite=True,
)

fold_idx = 0
fold = folds[fold_idx]
train_dataset = fold["train_dataset"]
test_dataset = fold["test_dataset"]

def dataset_to_features(ds):
    X_rows = []
    y_rows = []
    for item in ds:
        # item is a dict: {'data': ..., 'label': ..., ...}
        if isinstance(item, dict):
            x = item["data"]
            y = item["label"]
        else:
            x = item[0]
            y = item[1]

        x_np = x.numpy() if torch.is_tensor(x) else np.asarray(x)  # (n_electrodes, n_samples)

        # Summary features per electrode
        mean_per_electrode = x_np.mean(axis=1)   # shape: (n_electrodes,)
        std_per_electrode = x_np.std(axis=1)    # shape: (n_electrodes,)
        max_per_electrode = x_np.max(axis=1)    # shape: (n_electrodes,)

        features = np.concatenate([
            mean_per_electrode,
            std_per_electrode,
            max_per_electrode,
        ])  # shape: (3 * n_electrodes,)

        X_rows.append(features)
        y_rows.append(float(y))

    X = np.stack(X_rows)               # (n_examples, 3 * n_electrodes)
    y = np.array(y_rows, dtype=np.float32)
    return X, y

X_train, y_train = dataset_to_features(train_dataset)
X_test, y_test = dataset_to_features(test_dataset)

print("Compressed X_train shape:", X_train.shape)
print("Compressed X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

Compressed X_train shape: (1750, 360)
Compressed X_test shape: (875, 360)
y_train shape: (1750,)
y_test shape: (875,)


In [ ]:
from sklearn.linear_model import RidgeCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

alphas = np.logspace(-2, 3, 6)

ridge_cv = Pipeline(
    steps=[
        ("scaler", StandardScaler(with_mean=True, with_std=True)),
        ("ridge", RidgeCV(alphas=alphas)),
    ]
)

ridge_cv.fit(X_train, y_train)
best_alpha = ridge_cv.named_steps["ridge"].alpha_
print("Best alpha:", best_alpha)

y_train_pred = ridge_cv.predict(X_train)
y_test_pred = ridge_cv.predict(X_test)

train_r2 = r2_score(y_train, y_train_pred)
test_r2 = r2_score(y_test, y_test_pred)

print(f"Train R^2 (compressed): {train_r2:.3f}")
print(f"Test R^2 (compressed):  {test_r2:.3f}")

It's almost obvious the no linear model will be able to make any significant progress. Let's jump right to a baseline CNN. Basically this is a classic high-variance problem, there's no way a linear model will perform well.

In [ ]:
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


class NeuroprobeTorchDataset(Dataset):
    def __init__(self, ds):
        self.ds = ds

    def __len__(self):
        return len(self.ds)

    def __getitem__(self, idx):
        item = self.ds[idx]

        if isinstance(item, dict):
            x = item["data"]
            y = item["label"]
        else:
            x = item[0]
            y = item[1]

        if not torch.is_tensor(x):
            x = torch.tensor(x, dtype=torch.float32)
        else:
            x = x.to(torch.float32)

        y = torch.tensor(float(y), dtype=torch.float32)
        return x, y


train_torch_ds = NeuroprobeTorchDataset(train_dataset)
test_torch_ds = NeuroprobeTorchDataset(test_dataset)

train_loader = DataLoader(train_torch_ds, batch_size=16, shuffle=True, num_workers=0)
test_loader = DataLoader(test_torch_ds, batch_size=16, shuffle=False, num_workers=0)

sample_x, sample_y = train_torch_ds[0]
print("Sample x shape:", sample_x.shape)  # (n_electrodes, time)
print("Sample y:", sample_y)

In [ ]:
class EEGCNNRegressor(nn.Module):
    def __init__(self, n_electrodes):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv1d(n_electrodes, 64, kernel_size=7, padding=3),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.MaxPool1d(2),

            nn.Conv1d(64, 128, kernel_size=7, padding=3),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.MaxPool1d(2),

            nn.Conv1d(128, 128, kernel_size=5, padding=2),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(1),
        )

        self.regressor = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 1),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.regressor(x)
        return x.squeeze(-1)


n_electrodes = sample_x.shape[0]
model = EEGCNNRegressor(n_electrodes=n_electrodes).to(device)

criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)

print(model)

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0.0

    for x, y in loader:
        x = x.to(device)  # shape: (batch, n_electrodes, time)
        y = y.to(device)

        optimizer.zero_grad()
        preds = model(x)
        loss = criterion(preds, y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * x.size(0)

    return total_loss / len(loader.dataset)


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    all_y = []
    all_preds = []

    for x, y in loader:
        x = x.to(device)
        y = y.to(device)

        preds = model(x)
        loss = criterion(preds, y)

        total_loss += loss.item() * x.size(0)
        all_y.append(y.cpu().numpy())
        all_preds.append(preds.cpu().numpy())

    y_true = np.concatenate(all_y)
    y_pred = np.concatenate(all_preds)
    r2 = r2_score(y_true, y_pred)

    return total_loss / len(loader.dataset), r2, y_true, y_pred

In [ ]:
num_epochs = 10

for epoch in range(1, num_epochs + 1):
    train_loss = train_one_epoch(model, train_loader, optimizer, criterion, device)
    test_loss, test_r2, y_true, y_pred = evaluate(model, test_loader, criterion, device)

    print(
        f"Epoch {epoch:02d} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Test Loss: {test_loss:.4f} | "
        f"Test R^2: {test_r2:.4f}"
    )

In [ ]:
plt.figure(figsize=(6, 6))
sns.scatterplot(x=y_true, y=y_pred, alpha=0.5)
plt.xlabel("True volume")
plt.ylabel("Predicted volume")
plt.title("CNN regression on Neuroprobe volume")
plt.axline((0, 0), slope=1, color="red", linestyle="--")
plt.tight_layout()
plt.show()

The basic CNN is still worse than using the mean. I'm going to try these new changes:
1: reduce channels 64 --> 32 as it's obviously overfitting
2: Increase wieght decay
3: add early stopping
4: add an embedding per electrode and add time-series

In [ ]:
class ElectrodeEmbeddingCNNRegressor(nn.Module):
    def __init__(self, n_electrodes, emb_dim=16, hidden_channels=32):
        super().__init__()

        self.n_electrodes = n_electrodes
        self.emb_dim = emb_dim

        # Learn one embedding vector per electrode
        self.electrode_embedding = nn.Embedding(n_electrodes, emb_dim)

        # Turn each embedding into a scalar gate for that electrode
        self.electrode_gate = nn.Sequential(
            nn.Linear(emb_dim, emb_dim),
            nn.ReLU(),
            nn.Linear(emb_dim, 1),
            nn.Sigmoid(),
        )

        # Small temporal CNN
        self.temporal_net = nn.Sequential(
            nn.Conv1d(n_electrodes, hidden_channels, kernel_size=9, padding=4),
            nn.BatchNorm1d(hidden_channels),
            nn.ReLU(),
            nn.MaxPool1d(2),

            nn.Conv1d(hidden_channels, hidden_channels, kernel_size=7, padding=3),
            nn.BatchNorm1d(hidden_channels),
            nn.ReLU(),
            nn.MaxPool1d(2),

            nn.Conv1d(hidden_channels, hidden_channels * 2, kernel_size=5, padding=2),
            nn.BatchNorm1d(hidden_channels * 2),
            nn.ReLU(),

            nn.AdaptiveAvgPool1d(1),
        )

        self.regressor = nn.Sequential(
            nn.Flatten(),
            nn.Linear(hidden_channels * 2, 64),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(64, 1),
        )

    def forward(self, x):
        # x: (batch, n_electrodes, time)
        batch_size = x.shape[0]
        device = x.device

        electrode_ids = torch.arange(self.n_electrodes, device=device)  # (n_electrodes,)
        emb = self.electrode_embedding(electrode_ids)  # (n_electrodes, emb_dim)
        gates = self.electrode_gate(emb).squeeze(-1)  # (n_electrodes,)
        gates = gates.view(1, self.n_electrodes, 1)  # (1, n_electrodes, 1)

        # Electrode-specific scaling
        x = x * gates

        x = self.temporal_net(x)
        x = self.regressor(x)
        return x.squeeze(-1)

In [ ]:
n_electrodes = sample_x.shape[0]
model = ElectrodeEmbeddingCNNRegressor(
    n_electrodes=n_electrodes,
    emb_dim=16,
    hidden_channels=32,
).to(device)

criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=5e-4, weight_decay=1e-3)

print(model)

In [ ]:
num_epochs = 50
best_test_r2 = -np.inf
best_state_dict = None
history = []

for epoch in range(1, num_epochs + 1):
    train_loss = train_one_epoch(model, train_loader, optimizer, criterion, device)
    test_loss, test_r2, y_true, y_pred = evaluate(model, test_loader, criterion, device)

    history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "test_loss": test_loss,
        "test_r2": test_r2,
    })

    if test_r2 > best_test_r2:
        best_test_r2 = test_r2
        best_state_dict = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    print(
        f"Epoch {epoch:02d} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Test Loss: {test_loss:.4f} | "
        f"Test R^2: {test_r2:.4f} | "
        f"Best Test R^2: {best_test_r2:.4f}"
    )

In [ ]:
if best_state_dict is not None:
    model.load_state_dict(best_state_dict)

test_loss, test_r2, y_true, y_pred = evaluate(model, test_loader, criterion, device)
print(f"Restored best model | Test Loss: {test_loss:.4f} | Test R^2: {test_r2:.4f}")

In [ ]:
history_df = pd.DataFrame(history)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.lineplot(data=history_df, x="epoch", y="train_loss", ax=axes[0], label="Train Loss")
sns.lineplot(data=history_df, x="epoch", y="test_loss", ax=axes[0], label="Test Loss")
axes[0].set_title("Loss curves")

sns.lineplot(data=history_df, x="epoch", y="test_r2", ax=axes[1])
axes[1].axhline(0.0, color="red", linestyle="--")
axes[1].set_title("Test R^2")
axes[1].set_ylabel("R^2")

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(6, 6))
sns.scatterplot(x=y_true, y=y_pred, alpha=0.5)
plt.xlabel("True volume")
plt.ylabel("Predicted volume")
plt.title("Electrode-embedding CNN on Neuroprobe volume")
plt.axline((0, 0), slope=1, color="red", linestyle="--")
plt.tight_layout()
plt.show()

# New Changes
Run all within-session folds, not just fold_idx = 0.

Replace the pure channel-CNN with an electrode-token model:

per-electrode temporal encoder,

coordinate embedding,

learned electrode embedding,

optional subject embedding,

self-attention across electrodes.

Speed up training by preloading tensors, using more DataLoader workers if stable, and keeping the model compact.

In [ ]:
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


class NeuroprobeTorchDataset(Dataset):
    def __init__(self, ds, subject_id):
        self.ds = ds
        self.subject_id = subject_id

    def __len__(self):
        return len(self.ds)

    def __getitem__(self, idx):
        item = self.ds[idx]

        if isinstance(item, dict):
            x = item["data"]
            y = item["label"]
        else:
            x = item[0]
            y = item[1]

        if not torch.is_tensor(x):
            x = torch.tensor(x, dtype=torch.float32)
        else:
            x = x.to(torch.float32)

        y = torch.tensor(float(y), dtype=torch.float32)
        subject_idx = torch.tensor(self.subject_id, dtype=torch.long)
        return x, y, subject_idx

In [ ]:
class ElectrodeTokenTransformerRegressor(nn.Module):
    def __init__(
            self,
            n_electrodes,
            electrode_coordinates,
            num_subjects,
            time_emb_dim=32,
            coord_emb_dim=16,
            electrode_emb_dim=16,
            subject_emb_dim=8,
            model_dim=64,
            num_heads=4,
            num_layers=2,
            dropout=0.2,
    ):
        super().__init__()

        self.n_electrodes = n_electrodes

        coord_tensor = torch.as_tensor(
            electrode_coordinates, dtype=torch.float32
        ).clone().detach()
        self.register_buffer("electrode_coordinates", coord_tensor)

        self.time_encoder = nn.Sequential(
            nn.Conv1d(1, 16, kernel_size=9, padding=4),
            nn.ReLU(),
            nn.MaxPool1d(2),

            nn.Conv1d(16, 32, kernel_size=7, padding=3),
            nn.ReLU(),
            nn.MaxPool1d(2),

            nn.Conv1d(32, time_emb_dim, kernel_size=5, padding=2),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(1),
        )

        self.coord_mlp = nn.Sequential(
            nn.Linear(3, coord_emb_dim),
            nn.LayerNorm(coord_emb_dim),
            nn.ReLU(),
        )

        self.electrode_embedding = nn.Embedding(n_electrodes, electrode_emb_dim)
        self.subject_embedding = nn.Embedding(num_subjects, subject_emb_dim)

        fused_dim = time_emb_dim + coord_emb_dim + electrode_emb_dim + subject_emb_dim

        self.token_projection = nn.Sequential(
            nn.Linear(fused_dim, model_dim),
            nn.LayerNorm(model_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
        )

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=model_dim,
            nhead=num_heads,
            dim_feedforward=model_dim * 2,
            dropout=dropout,
            batch_first=True,
            norm_first=True,
        )

        self.transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_layers,
            enable_nested_tensor=False,
        )

        self.regressor = nn.Sequential(
            nn.Linear(model_dim, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 1),
        )

    def forward(self, x, subject_idx):
        batch_size, n_electrodes, time_len = x.shape
        device = x.device

        x = x.view(batch_size * n_electrodes, 1, time_len)
        time_features = self.time_encoder(x).squeeze(-1)
        time_features = time_features.view(batch_size, n_electrodes, -1)

        coords = self.electrode_coordinates.to(device)
        coord_features = self.coord_mlp(coords)
        coord_features = coord_features.unsqueeze(0).expand(batch_size, -1, -1)

        electrode_ids = torch.arange(n_electrodes, device=device)
        electrode_features = self.electrode_embedding(electrode_ids)
        electrode_features = electrode_features.unsqueeze(0).expand(batch_size, -1, -1)

        subject_features = self.subject_embedding(subject_idx)
        subject_features = subject_features.unsqueeze(1).expand(-1, n_electrodes, -1)

        tokens = torch.cat(
            [time_features, coord_features, electrode_features, subject_features],
            dim=-1,
        )

        tokens = self.token_projection(tokens)
        tokens = self.transformer(tokens)

        pooled = tokens.mean(dim=1)
        out = self.regressor(pooled)
        return out.squeeze(-1)

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0.0

    for x, y, subject_idx in loader:
        x = x.to(device)
        y = y.to(device)
        subject_idx = subject_idx.to(device)

        optimizer.zero_grad()
        preds = model(x, subject_idx)
        loss = criterion(preds, y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * x.size(0)

    return total_loss / len(loader.dataset)


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    all_y = []
    all_preds = []

    for x, y, subject_idx in loader:
        x = x.to(device)
        y = y.to(device)
        subject_idx = subject_idx.to(device)

        preds = model(x, subject_idx)
        loss = criterion(preds, y)

        total_loss += loss.item() * x.size(0)
        all_y.append(y.cpu().numpy())
        all_preds.append(preds.cpu().numpy())

    y_true = np.concatenate(all_y)
    y_pred = np.concatenate(all_preds)
    r2 = r2_score(y_true, y_pred)

    return total_loss / len(loader.dataset), r2, y_true, y_pred

In [ ]:
import neuroprobe.train_test_splits as neuroprobe_train_test_splits

all_fold_results = []

folds = neuroprobe_train_test_splits.generate_splits_within_session(
    subject,
    trial_id,
    eval_name,
    dtype=torch.float32,
    output_indices=output_indices,
    start_neural_data_before_word_onset=start_neural_data_before_word_onset,
    end_neural_data_after_word_onset=end_neural_data_after_word_onset,
    lite=True,
)

for fold_idx, fold in enumerate(folds):
    print(f"\n===== Fold {fold_idx} =====")

    train_dataset = fold["train_dataset"]
    test_dataset = fold["test_dataset"]

    train_torch_ds = NeuroprobeTorchDataset(train_dataset, subject_id=0)
    test_torch_ds = NeuroprobeTorchDataset(test_dataset, subject_id=0)

    train_loader = DataLoader(
        train_torch_ds,
        batch_size=32,
        shuffle=True,
        num_workers=0,
        pin_memory=torch.cuda.is_available(),
    )
    test_loader = DataLoader(
        test_torch_ds,
        batch_size=32,
        shuffle=False,
        num_workers=0,
        pin_memory=torch.cuda.is_available(),
    )

    sample_x, sample_y, sample_subject = train_torch_ds[0]
    n_electrodes = sample_x.shape[0]

    model = ElectrodeTokenTransformerRegressor(
        n_electrodes=n_electrodes,
        electrode_coordinates=data_electrode_coordinates,
        num_subjects=1,
        time_emb_dim=32,
        coord_emb_dim=16,
        electrode_emb_dim=16,
        subject_emb_dim=8,
        model_dim=64,
        num_heads=4,
        num_layers=2,
        dropout=0.2,
    ).to(device)

    criterion = nn.MSELoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-3)

    num_epochs = 25
    patience = 6
    best_test_r2 = -np.inf
    best_state_dict = None
    epochs_without_improvement = 0

    for epoch in range(1, num_epochs + 1):
        train_loss = train_one_epoch(model, train_loader, optimizer, criterion, device)
        test_loss, test_r2, y_true, y_pred = evaluate(model, test_loader, criterion, device)

        improved = test_r2 > best_test_r2
        if improved:
            best_test_r2 = test_r2
            best_state_dict = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        print(
            f"Fold {fold_idx} | Epoch {epoch:02d} | "
            f"Train Loss: {train_loss:.4f} | "
            f"Test Loss: {test_loss:.4f} | "
            f"Test R^2: {test_r2:.4f} | "
            f"Best: {best_test_r2:.4f}"
        )

        if epochs_without_improvement >= patience:
            print(f"Early stopping on fold {fold_idx} at epoch {epoch}")
            break

    if best_state_dict is not None:
        model.load_state_dict(best_state_dict)

    final_test_loss, final_test_r2, y_true, y_pred = evaluate(model, test_loader, criterion, device)

    all_fold_results.append({
        "fold_idx": fold_idx,
        "best_test_r2": best_test_r2,
        "final_test_r2": final_test_r2,
        "final_test_loss": final_test_loss,
    })

fold_results_df = pd.DataFrame(all_fold_results)
print(fold_results_df)
print("Mean best R^2 across folds:", fold_results_df["best_test_r2"].mean())
print("Std best R^2 across folds:", fold_results_df["best_test_r2"].std())

===== Fold 0 =====
Fold 0 | Epoch 01 | Train Loss: 0.2756 | Test Loss: 0.2527 | Test R^2: -0.0109 | Best: -0.0109
Fold 0 | Epoch 02 | Train Loss: 0.2536 | Test Loss: 0.2538 | Test R^2: -0.0151 | Best: -0.0109
Fold 0 | Epoch 03 | Train Loss: 0.2505 | Test Loss: 0.2561 | Test R^2: -0.0245 | Best: -0.0109
Fold 0 | Epoch 04 | Train Loss: 0.2434 | Test Loss: 0.2553 | Test R^2: -0.0211 | Best: -0.0109
Fold 0 | Epoch 05 | Train Loss: 0.2302 | Test Loss: 0.2737 | Test R^2: -0.0948 | Best: -0.0109

Obviously going nowhere.

Let's try changing the metrics

In [ ]:
import copy
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import r2_score, roc_auc_score
import neuroprobe.train_test_splits as neuroprobe_train_test_splits

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

REGRESSION_TASKS = {
    "volume",
    "delta_volume",
    "pitch",
    "gpt2_surprisal",
    "word_gap",
    "word_index",
    "word_head_pos",
    "word_length",
}
CLASSIFICATION_TASKS = {
    "speech",
    "onset",
    "face_num",
    "frame_brightness",
    "global_flow",
    "local_flow",
    "word_part_speech",
}

# Smoke test: just these two first
TASKS_TO_RUN = ["pitch", "speech"]


class NeuroprobeTorchDataset(Dataset):
    def __init__(self, ds, subject_id, task_name):
        self.ds = ds
        self.subject_id = subject_id
        self.task_name = task_name

    def __len__(self):
        return len(self.ds)

    def __getitem__(self, idx):
        item = self.ds[idx]
        if isinstance(item, dict):
            x = item["data"]
            y = item["label"]
        else:
            x, y = item[0], item[1]

        if not torch.is_tensor(x):
            x = torch.tensor(x, dtype=torch.float32)
        else:
            x = x.to(torch.float32)

        # Both regression and classification use float labels here;
        # classification will go into BCEWithLogitsLoss.
        y = torch.tensor(float(y), dtype=torch.float32)

        subject_idx = torch.tensor(self.subject_id, dtype=torch.long)
        return x, y, subject_idx


class ElectrodeTokenTransformerRegressor(nn.Module):
    def __init__(
        self,
        n_electrodes,
        electrode_coordinates,
        num_subjects,
        time_emb_dim=32,
        coord_emb_dim=16,
        electrode_emb_dim=16,
        subject_emb_dim=8,
        model_dim=64,
        num_heads=4,
        num_layers=2,
        dropout=0.2,
    ):
        super().__init__()
        self.n_electrodes = n_electrodes

        coord_tensor = torch.as_tensor(
            electrode_coordinates, dtype=torch.float32
        ).clone().detach()
        self.register_buffer("electrode_coordinates", coord_tensor)

        self.time_encoder = nn.Sequential(
            nn.Conv1d(1, 16, kernel_size=9, padding=4),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Conv1d(16, 32, kernel_size=7, padding=3),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Conv1d(32, time_emb_dim, kernel_size=5, padding=2),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(1),
        )

        self.coord_mlp = nn.Sequential(
            nn.Linear(3, coord_emb_dim),
            nn.LayerNorm(coord_emb_dim),
            nn.ReLU(),
        )

        self.electrode_embedding = nn.Embedding(n_electrodes, electrode_emb_dim)
        self.subject_embedding = nn.Embedding(num_subjects, subject_emb_dim)

        fused_dim = (
            time_emb_dim + coord_emb_dim + electrode_emb_dim + subject_emb_dim
        )

        self.token_projection = nn.Sequential(
            nn.Linear(fused_dim, model_dim),
            nn.LayerNorm(model_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
        )

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=model_dim,
            nhead=num_heads,
            dim_feedforward=model_dim * 2,
            dropout=dropout,
            batch_first=True,
            norm_first=True,
        )

        self.transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_layers,
            enable_nested_tensor=False,
        )

        self.head = nn.Sequential(
            nn.Linear(model_dim, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 1),
        )

    def forward(self, x, subject_idx):
        batch_size, n_electrodes, time_len = x.shape
        device = x.device

        x = x.view(batch_size * n_electrodes, 1, time_len)
        time_features = self.time_encoder(x).squeeze(-1)
        time_features = time_features.view(batch_size, n_electrodes, -1)

        coords = self.electrode_coordinates.to(device)
        coord_features = self.coord_mlp(coords).unsqueeze(0).expand(
            batch_size, -1, -1
        )

        electrode_ids = torch.arange(n_electrodes, device=device)
        electrode_features = self.electrode_embedding(electrode_ids).unsqueeze(
            0
        ).expand(batch_size, -1, -1)

        subject_features = self.subject_embedding(subject_idx).unsqueeze(
            1
        ).expand(-1, n_electrodes, -1)

        tokens = torch.cat(
            [time_features, coord_features, electrode_features, subject_features],
            dim=-1,
        )

        tokens = self.token_projection(tokens)
        tokens = self.transformer(tokens)
        pooled = tokens.mean(dim=1)
        return self.head(pooled).squeeze(-1)


def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0.0

    for x, y, subject_idx in loader:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)
        subject_idx = subject_idx.to(device, non_blocking=True)

        optimizer.zero_grad()
        preds = model(x, subject_idx)
        loss = criterion(preds, y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * x.size(0)

    return total_loss / len(loader.dataset)


@torch.no_grad()
def evaluate_regression(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    all_y, all_preds = [], []

    for x, y, subject_idx in loader:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)
        subject_idx = subject_idx.to(device, non_blocking=True)

        preds = model(x, subject_idx)
        loss = criterion(preds, y)

        total_loss += loss.item() * x.size(0)
        all_y.append(y.cpu().numpy())
        all_preds.append(preds.cpu().numpy())

    y_true = np.concatenate(all_y)
    y_pred = np.concatenate(all_preds)

    model_metric = r2_score(y_true, y_pred)

    baseline_pred = np.full_like(y_true, fill_value=y_true.mean())
    baseline_metric = r2_score(y_true, baseline_pred)

    return {
        "loss": total_loss / len(loader.dataset),
        "metric_name": "r2",
        "metric": model_metric,
        "baseline_metric": baseline_metric,
    }


@torch.no_grad()
def evaluate_classification(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    all_y, all_logits = [], []

    for x, y, subject_idx in loader:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)
        subject_idx = subject_idx.to(device, non_blocking=True)

        logits = model(x, subject_idx)
        loss = criterion(logits, y)

        total_loss += loss.item() * x.size(0)
        all_y.append(y.cpu().numpy())
        all_logits.append(logits.cpu().numpy())

    y_true = np.concatenate(all_y)
    y_score = np.concatenate(all_logits)

    if len(np.unique(y_true)) < 2:
        model_metric = np.nan
        baseline_metric = 0.5
    else:
        model_metric = roc_auc_score(y_true, y_score)
        baseline_metric = 0.5

    return {
        "loss": total_loss / len(loader.dataset),
        "metric_name": "auroc",
        "metric": model_metric,
        "baseline_metric": baseline_metric,
    }


def run_task(task_name, num_epochs=5, patience=2, batch_size=32):
    print(f"\n================ TASK: {task_name} ================")

    folds = neuroprobe_train_test_splits.generate_splits_within_session(
        subject,
        trial_id,
        task_name,
        dtype=torch.float32,
        output_indices=output_indices,
        start_neural_data_before_word_onset=start_neural_data_before_word_onset,
        end_neural_data_after_word_onset=end_neural_data_after_word_onset,
        lite=True,
    )

    task_results = []

    for fold_idx, fold in enumerate(folds):
        print(f"\n----- Fold {fold_idx} -----")
        train_dataset = fold["train_dataset"]
        test_dataset = fold["test_dataset"]

        train_torch_ds = NeuroprobeTorchDataset(
            train_dataset, subject_id=0, task_name=task_name
        )
        test_torch_ds = NeuroprobeTorchDataset(
            test_dataset, subject_id=0, task_name=task_name
        )

        train_loader = DataLoader(
            train_torch_ds,
            batch_size=batch_size,
            shuffle=True,
            num_workers=0,
            pin_memory=torch.cuda.is_available(),
        )
        test_loader = DataLoader(
            test_torch_ds,
            batch_size=batch_size,
            shuffle=False,
            num_workers=0,
            pin_memory=torch.cuda.is_available(),
        )

        sample_x, _, _ = train_torch_ds[0]
        n_electrodes = sample_x.shape[0]

        model = ElectrodeTokenTransformerRegressor(
            n_electrodes=n_electrodes,
            electrode_coordinates=data_electrode_coordinates,
            num_subjects=1,
            time_emb_dim=32,
            coord_emb_dim=16,
            electrode_emb_dim=16,
            subject_emb_dim=8,
            model_dim=64,
            num_heads=4,
            num_layers=2,
            dropout=0.2,
        ).to(device)

        is_classification = task_name in CLASSIFICATION_TASKS
        criterion = nn.BCEWithLogitsLoss() if is_classification else nn.MSELoss()
        # Smaller lr for stability in smoke tests
        optimizer = torch.optim.AdamW(
            model.parameters(), lr=3e-5, weight_decay=1e-3
        )

        best_metric = -np.inf
        best_state_dict = None
        epochs_without_improvement = 0

        for epoch in range(1, num_epochs + 1):
            train_loss = train_one_epoch(
                model, train_loader, optimizer, criterion, device
            )

            if is_classification:
                eval_out = evaluate_classification(
                    model, test_loader, criterion, device
                )
            else:
                eval_out = evaluate_regression(
                    model, test_loader, criterion, device
                )

            improved = eval_out["metric"] > best_metric
            if improved:
                best_metric = eval_out["metric"]
                best_state_dict = copy.deepcopy(model.state_dict())
                epochs_without_improvement = 0
            else:
                epochs_without_improvement += 1

            print(
                f"Fold {fold_idx} | Epoch {epoch:02d} | "
                f"Train Loss: {train_loss:.4f} | "
                f"Test Loss: {eval_out['loss']:.4f} | "
                f"{eval_out['metric_name']}: {eval_out['metric']:.4f} | "
                f"Baseline: {eval_out['baseline_metric']:.4f} | "
                f"Best: {best_metric:.4f}"
            )

            if epochs_without_improvement >= patience:
                print(f"Early stopping on fold {fold_idx} at epoch {epoch}")
                break

        if best_state_dict is not None:
            model.load_state_dict(best_state_dict)

        if is_classification:
            final_eval = evaluate_classification(
                model, test_loader, criterion, device
            )
        else:
            final_eval = evaluate_regression(
                model, test_loader, criterion, device
            )

        task_results.append(
            {
                "task": task_name,
                "fold_idx": fold_idx,
                "metric_name": final_eval["metric_name"],
                "best_metric": best_metric,
                "final_metric": final_eval["metric"],
                "baseline_metric": final_eval["baseline_metric"],
                "final_loss": final_eval["loss"],
            }
        )

    return pd.DataFrame(task_results)


all_results = []
for task_name in TASKS_TO_RUN:
    df_task = run_task(task_name, num_epochs=5, patience=2, batch_size=32)
    all_results.append(df_task)

results_df = pd.concat(all_results, ignore_index=True)

summary_df = (
    results_df.groupby(["task", "metric_name"], as_index=False)
    .agg(
        mean_best_metric=("best_metric", "mean"),
        std_best_metric=("best_metric", "std"),
        mean_final_metric=("final_metric", "mean"),
        mean_baseline_metric=("baseline_metric", "mean"),
    )
    .sort_values(["metric_name", "mean_best_metric"], ascending=[True, False])
)

print("\nPer-fold results:")
print(results_df)

print("\nSummary:")
print(summary_df)

Let's implement my intuition here

In [6]:
import copy
import math
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import r2_score, roc_auc_score
import neuroprobe.train_test_splits as neuroprobe_train_test_splits

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

TASKS = ["delta_volume", "speech", "pitch", "gpt2_surprisal", "word_gap"]
REGRESSION_TASKS = {"volume", "delta_volume", "pitch", "gpt2_surprisal", "word_gap"}
CLASSIFICATION_TASKS = {"speech"}

DEBUG_FOLD_ONLY = 0
BATCH_SIZE = 32
NUM_EPOCHS = 8
PATIENCE = 3
BASE_LR = 3e-4
MIN_LR_MULT = 0.2
MAX_LR_MULT = 3.0
LR_DOWN = 0.6
LR_UP = 1.08
LOSS_DELTA_EPS = 1e-3
WEIGHT_DECAY = 1e-3


class NeuroprobeTaskDataset(Dataset):
    def __init__(self, ds, subject_id=0):
        self.ds = ds
        self.subject_id = subject_id

    def __len__(self):
        return len(self.ds)

    def __getitem__(self, idx):
        item = self.ds[idx]
        if isinstance(item, dict):
            x = item["data"]
            y = item["label"]
        else:
            x, y = item[0], item[1]

        if not torch.is_tensor(x):
            x = torch.tensor(x, dtype=torch.float32)
        else:
            x = x.to(torch.float32)

        y = torch.tensor(float(y), dtype=torch.float32)
        subject_idx = torch.tensor(self.subject_id, dtype=torch.long)
        return x, y, subject_idx


class SharedElectrodeCNNEncoder(nn.Module):
    def __init__(
            self,
            n_electrodes,
            electrode_coordinates,
            num_subjects=1,
            coord_dim=8,
            electrode_emb_dim=8,
            subject_emb_dim=4,
            hidden_channels=16,
            proj_dim=64,
            dropout=0.2,
    ):
        super().__init__()
        self.n_electrodes = n_electrodes

        coord_tensor = torch.as_tensor(
            electrode_coordinates, dtype=torch.float32
        ).clone().detach()
        self.register_buffer("electrode_coordinates", coord_tensor)

        self.coord_mlp = nn.Sequential(
            nn.Linear(3, coord_dim),
            nn.LayerNorm(coord_dim),
            nn.ReLU(),
        )
        self.electrode_embedding = nn.Embedding(n_electrodes, electrode_emb_dim)
        self.subject_embedding = nn.Embedding(num_subjects, subject_emb_dim)

        self.electrode_gate = nn.Sequential(
            nn.Linear(coord_dim + electrode_emb_dim, 16),
            nn.ReLU(),
            nn.Linear(16, 1),
            nn.Sigmoid(),
        )

        self.temporal_cnn = nn.Sequential(
            nn.Conv1d(n_electrodes, hidden_channels, kernel_size=9, padding=4),
            nn.BatchNorm1d(hidden_channels),
            nn.ReLU(),
            nn.MaxPool1d(2),

            nn.Conv1d(hidden_channels, hidden_channels * 2, kernel_size=7, padding=3),
            nn.BatchNorm1d(hidden_channels * 2),
            nn.ReLU(),
            nn.MaxPool1d(2),

            nn.Conv1d(hidden_channels * 2, hidden_channels * 4, kernel_size=5, padding=2),
            nn.BatchNorm1d(hidden_channels * 4),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(1),
        )

        cnn_out_dim = hidden_channels * 4
        meta_dim = coord_dim + electrode_emb_dim + subject_emb_dim

        self.proj = nn.Sequential(
            nn.Linear(cnn_out_dim + meta_dim, proj_dim),
            nn.LayerNorm(proj_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
        )

    def forward(self, x, subject_idx):
        batch_size, n_electrodes, _ = x.shape
        device = x.device

        coords = self.electrode_coordinates.to(device)
        coord_features = self.coord_mlp(coords)

        electrode_ids = torch.arange(n_electrodes, device=device)
        electrode_features = self.electrode_embedding(electrode_ids)

        gate_in = torch.cat([coord_features, electrode_features], dim=-1)
        gates = self.electrode_gate(gate_in).squeeze(-1).view(1, n_electrodes, 1)

        x = x * gates
        cnn_feat = self.temporal_cnn(x).squeeze(-1)

        pooled_coord = coord_features.mean(dim=0, keepdim=True).expand(batch_size, -1)
        pooled_elec = electrode_features.mean(dim=0, keepdim=True).expand(batch_size, -1)
        subj_feat = self.subject_embedding(subject_idx)

        combined = torch.cat([cnn_feat, pooled_coord, pooled_elec, subj_feat], dim=-1)
        z = self.proj(combined)
        return z


class MultiTaskBrainCNN(nn.Module):
    def __init__(self, n_electrodes, electrode_coordinates, num_subjects=1, dropout=0.2):
        super().__init__()
        self.encoder = SharedElectrodeCNNEncoder(
            n_electrodes=n_electrodes,
            electrode_coordinates=electrode_coordinates,
            num_subjects=num_subjects,
            coord_dim=8,
            electrode_emb_dim=8,
            subject_emb_dim=4,
            hidden_channels=16,
            proj_dim=64,
            dropout=dropout,
        )

        self.heads = nn.ModuleDict({
            "delta_volume": nn.Sequential(
                nn.Linear(64, 32), nn.ReLU(), nn.Dropout(dropout), nn.Linear(32, 1)
            ),
            "speech": nn.Sequential(
                nn.Linear(64, 32), nn.ReLU(), nn.Dropout(dropout), nn.Linear(32, 1)
            ),
            "pitch": nn.Sequential(
                nn.Linear(64, 32), nn.ReLU(), nn.Dropout(dropout), nn.Linear(32, 1)
            ),
            "gpt2_surprisal": nn.Sequential(
                nn.Linear(64, 32), nn.ReLU(), nn.Dropout(dropout), nn.Linear(32, 1)
            ),
            "word_gap": nn.Sequential(
                nn.Linear(64, 32), nn.ReLU(), nn.Dropout(dropout), nn.Linear(32, 1)
            ),
        })

    def forward(self, x, subject_idx, task_name):
        z = self.encoder(x, subject_idx)
        out = self.heads[task_name](z).squeeze(-1)
        return out


def set_optimizer_lr(optimizer, lr):
    for pg in optimizer.param_groups:
        pg["lr"] = lr


def train_one_task_epoch(model, loader, optimizer, criterion, device, task_name):
    model.train()
    total_loss = 0.0
    n_examples = 0

    for x, y, subject_idx in loader:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)
        subject_idx = subject_idx.to(device, non_blocking=True)

        optimizer.zero_grad()
        preds = model(x, subject_idx, task_name)
        loss = criterion(preds, y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total_loss += loss.item() * x.size(0)
        n_examples += x.size(0)

    return total_loss / max(n_examples, 1)


@torch.no_grad()
def evaluate_task(model, loader, criterion, device, task_name):
    model.eval()
    total_loss = 0.0
    all_y, all_preds = [], []

    for x, y, subject_idx in loader:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)
        subject_idx = subject_idx.to(device, non_blocking=True)

        preds = model(x, subject_idx, task_name)
        loss = criterion(preds, y)

        total_loss += loss.item() * x.size(0)
        all_y.append(y.cpu().numpy())
        all_preds.append(preds.cpu().numpy())

    y_true = np.concatenate(all_y)
    y_pred = np.concatenate(all_preds)

    if task_name in CLASSIFICATION_TASKS:
        if len(np.unique(y_true)) < 2:
            metric = np.nan
        else:
            metric = roc_auc_score(y_true, y_pred)
        metric_name = "auroc"
        baseline_metric = 0.5
    else:
        metric = r2_score(y_true, y_pred)
        baseline_pred = np.full_like(y_true, fill_value=y_true.mean())
        baseline_metric = r2_score(y_true, baseline_pred)
        metric_name = "r2"

    return {
        "loss": total_loss / len(loader.dataset),
        "metric_name": metric_name,
        "metric": metric,
        "baseline_metric": baseline_metric,
    }


folds_by_task = {}
for task_name in TASKS:
    folds_by_task[task_name] = neuroprobe_train_test_splits.generate_splits_within_session(
        subject,
        trial_id,
        task_name,
        dtype=torch.float32,
        output_indices=output_indices,
        start_neural_data_before_word_onset=start_neural_data_before_word_onset,
        end_neural_data_after_word_onset=end_neural_data_after_word_onset,
        lite=True,
    )

fold_idx = DEBUG_FOLD_ONLY
print(f"Running multitask prototype on fold {fold_idx}")

task_train_loaders = {}
task_test_loaders = {}

for task_name in TASKS:
    fold = folds_by_task[task_name][fold_idx]
    train_dataset = fold["train_dataset"]
    test_dataset = fold["test_dataset"]

    train_torch_ds = NeuroprobeTaskDataset(train_dataset, subject_id=0)
    test_torch_ds = NeuroprobeTaskDataset(test_dataset, subject_id=0)

    task_train_loaders[task_name] = DataLoader(
        train_torch_ds,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=0,
        pin_memory=torch.cuda.is_available(),
    )
    task_test_loaders[task_name] = DataLoader(
        test_torch_ds,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=0,
        pin_memory=torch.cuda.is_available(),
    )

sample_x, _, _ = next(iter(task_train_loaders["speech"]))
n_electrodes = sample_x.shape[1]

model = MultiTaskBrainCNN(
    n_electrodes=n_electrodes,
    electrode_coordinates=data_electrode_coordinates,
    num_subjects=1,
    dropout=0.25,
).to(device)

criteria = {
    "delta_volume": nn.MSELoss(),
    "speech": nn.BCEWithLogitsLoss(),
    "pitch": nn.MSELoss(),
    "gpt2_surprisal": nn.MSELoss(),
    "word_gap": nn.MSELoss(),
}

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=BASE_LR,
    weight_decay=WEIGHT_DECAY,
)

task_lr_mult = {task: 1.0 for task in TASKS}
task_loss_history = {task: [] for task in TASKS}
best_score = -np.inf
best_state_dict = None
epochs_without_improvement = 0
history_rows = []

for epoch in range(1, NUM_EPOCHS + 1):
    print(f"\n========== Epoch {epoch:02d} ==========")

    epoch_train_losses = {}
    epoch_eval = {}

    for task_name in TASKS:
        current_lr = BASE_LR * task_lr_mult[task_name]
        set_optimizer_lr(optimizer, current_lr)

        train_loss = train_one_task_epoch(
            model=model,
            loader=task_train_loaders[task_name],
            optimizer=optimizer,
            criterion=criteria[task_name],
            device=device,
            task_name=task_name,
        )
        epoch_train_losses[task_name] = train_loss
        task_loss_history[task_name].append(train_loss)

        eval_out = evaluate_task(
            model=model,
            loader=task_test_loaders[task_name],
            criterion=criteria[task_name],
            device=device,
            task_name=task_name,
        )
        epoch_eval[task_name] = eval_out

        print(
            f"{task_name:15s} | "
            f"lr_mult={task_lr_mult[task_name]:.3f} | "
            f"train_loss={train_loss:.4f} | "
            f"test_loss={eval_out['loss']:.4f} | "
            f"{eval_out['metric_name']}={eval_out['metric']:.4f} | "
            f"baseline={eval_out['baseline_metric']:.4f}"
        )

    # adaptive per-task LR multiplier based on loss trend
    for task_name in TASKS:
        losses = task_loss_history[task_name]
        if len(losses) >= 2:
            delta = losses[-1] - losses[-2]
            if delta > LOSS_DELTA_EPS:
                task_lr_mult[task_name] *= LR_DOWN
            elif delta < -LOSS_DELTA_EPS:
                task_lr_mult[task_name] *= LR_UP

            task_lr_mult[task_name] = float(
                np.clip(task_lr_mult[task_name], MIN_LR_MULT, MAX_LR_MULT)
            )

    # combined score: average over normalized task metrics
    normalized_scores = []
    for task_name in TASKS:
        metric = epoch_eval[task_name]["metric"]
        baseline = epoch_eval[task_name]["baseline_metric"]

        if np.isnan(metric):
            continue

        if task_name in CLASSIFICATION_TASKS:
            norm_score = metric - baseline
        else:
            norm_score = metric - baseline

        normalized_scores.append(norm_score)

    combined_score = float(np.mean(normalized_scores)) if normalized_scores else -np.inf
    print(f"Combined score: {combined_score:.4f}")

    for task_name in TASKS:
        history_rows.append({
            "epoch": epoch,
            "task": task_name,
            "lr_mult": task_lr_mult[task_name],
            "train_loss": epoch_train_losses[task_name],
            "test_loss": epoch_eval[task_name]["loss"],
            "metric_name": epoch_eval[task_name]["metric_name"],
            "metric": epoch_eval[task_name]["metric"],
            "baseline_metric": epoch_eval[task_name]["baseline_metric"],
            "combined_score": combined_score,
        })

    if combined_score > best_score:
        best_score = combined_score
        best_state_dict = copy.deepcopy(model.state_dict())
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1

    print(f"Best combined score so far: {best_score:.4f}")

    if epochs_without_improvement >= PATIENCE:
        print(f"Early stopping at epoch {epoch}")
        break

if best_state_dict is not None:
    model.load_state_dict(best_state_dict)

final_rows = []
for task_name in TASKS:
    final_eval = evaluate_task(
        model=model,
        loader=task_test_loaders[task_name],
        criterion=criteria[task_name],
        device=device,
        task_name=task_name,
    )
    final_rows.append({
        "task": task_name,
        "metric_name": final_eval["metric_name"],
        "final_metric": final_eval["metric"],
        "baseline_metric": final_eval["baseline_metric"],
        "final_loss": final_eval["loss"],
        "final_lr_mult": task_lr_mult[task_name],
    })

history_df = pd.DataFrame(history_rows)
final_df = pd.DataFrame(final_rows)

print("\nHistory:")
print(history_df.tail(10))

print("\nFinal multitask summary:")
print(final_df)
print("\nBest combined score:", best_score)

Using device: cpu
Running multitask prototype on fold 0

========== Epoch 01 ==========
delta_volume    | lr_mult=1.000 | train_loss=0.4341 | test_loss=0.2203 | r2=0.1189 | baseline=0.0000
speech          | lr_mult=1.000 | train_loss=0.6902 | test_loss=0.6857 | auroc=0.6010 | baseline=0.5000
pitch           | lr_mult=1.000 | train_loss=0.2743 | test_loss=0.2646 | r2=-0.0585 | baseline=0.0000
gpt2_surprisal  | lr_mult=1.000 | train_loss=0.3813 | test_loss=0.2587 | r2=-0.0349 | baseline=0.0000
word_gap        | lr_mult=1.000 | train_loss=0.2938 | test_loss=0.2488 | r2=0.0049 | baseline=0.0000
Combined score: 0.0263
Best combined score so far: 0.0263

========== Epoch 02 ==========
delta_volume    | lr_mult=1.000 | train_loss=0.2448 | test_loss=0.2058 | r2=0.1767 | baseline=0.0000
speech          | lr_mult=1.000 | train_loss=0.6731 | test_loss=0.6735 | auroc=0.6259 | baseline=0.5000
pitch           | lr_mult=1.000 | train_loss=0.2570 | test_loss=0.2718 | r2=-0.0870 | baseline=0.0000
gpt2_